0) Imports

In [42]:
import pandas as pd
import numpy as np
from PTF_construction import build_constant_prepay_ptf, build_hybrid_ptf
from pricing import evaluate_portfolio_black76
from plotting import plot_bands

ImportError: cannot import name 'plot_bands' from 'plotting' (/Users/niklasmoelders/projects/swaptions_PTF/plotting.py)

1) Set Parameters

In [39]:
# 1. Setting
date = pd.to_datetime('2026-01-01')
N_0 = 100

# 2. Mortgage Data
maturity_years = 30
payment_freq = 12
CPR_exp = 0.1
CPR_max = 0.15
CPR_min = 0.01                              # General Baseline
mortgage_rate = 0.05
shock = 0.02                                # Basel III

# 3. Market Data (Toy Example)

vol=0.30                                                        # Constant Vol
tenors = np.array([1.0, 5.0, 10.0, 15.0, 20.0])
z_curve = np.array([0.01, 0.015, 0.02, 0.022, 0.025])
strike_rate = 0.02                                              # Strike of ATM Swaptions is typically Par rate

2. Build The Constant Portfolio

In [35]:
constant_PTF = build_constant_prepay_ptf(date, maturity_years, payment_freq, N_0, CPR_exp, mortgage_rate, strike_rate, CPR_min=0.01)
constant_PTF.head()

,OPTION_EXPIRY,SWAP_TENOR,STRIKE,NOTIONAL,TYPE
0,2026-02-01,29.91,0.02,0.789494,RECEIVER_ATM
1,2026-03-01,29.84,0.02,0.780694,RECEIVER_ATM
2,2026-04-01,29.75,0.02,0.771968,RECEIVER_ATM
3,2026-05-01,29.67,0.02,0.763316,RECEIVER_ATM
4,2026-06-01,29.59,0.02,0.754737,RECEIVER_ATM


3. Build the Hybrid Portfolio

In [36]:
hybrid_PTF = build_hybrid_ptf(date, maturity_years, payment_freq, N_0, CPR_exp, CPR_max, mortgage_rate, strike_rate, shock, CPR_min=0.01)
hybrid_PTF.head()

,OPTION_EXPIRY,SWAP_TENOR,STRIKE,NOTIONAL,TYPE
0,2026-02-01,29.91,0.02,0.789494,RECEIVER_ATM
1,2026-02-01,29.91,0.01,0.470468,RECEIVER_OTM
2,2026-03-01,29.84,0.02,0.780694,RECEIVER_ATM
3,2026-03-01,29.84,0.01,0.459296,RECEIVER_OTM
4,2026-04-01,29.75,0.02,0.771968,RECEIVER_ATM


In [37]:
print(f"The Hybrid Portfolio Contains {len(hybrid_PTF[hybrid_PTF['TYPE'] == 'RECEIVER_OTM'])} OTM Swaptions and {len(hybrid_PTF[hybrid_PTF['TYPE'] == 'RECEIVER_ATM'])} ATM Swaptions")

The Hybrid Portfolio Contains 78 OTM Swaptions and 154 ATM Swaptions


4) Price the Portfolios

In [40]:
constant_price = evaluate_portfolio_black76(constant_PTF, date, z_curve, tenors, vol=0.30)
hybrid_price = evaluate_portfolio_black76(hybrid_PTF, date, z_curve, tenors, vol=0.30)
print(f'Constant Portfolio Price is {constant_price:.2f}% of N0')
print(f'Hybrid Portfolio Price is {hybrid_price:.2f}% of N0')

Constant Portfolio Price is 4.08% of N0
Hybrid Portfolio Price is 4.13% of N0
